### Data compression
Import libraries

In [ ]:
import sys

libs = {
  "./../modules",
  "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [ ]:
import os
import json
import numpy as np
import dill as pickle
import dask.array as da

In [ ]:
import rom as rom_mod
import plotting as plt_mod
import reading as read_mod

Define inputs

In [ ]:
# Path to saved solution
path_to_sol = "./../output/data/dset.p"
# Path to saving folder
path_to_save = "./../output/compress/"
# Maximum number of reduced dimensions
rdim_max = 10

In [ ]:
os.makedirs(path_to_save, exist_ok=True)

Read saved solution

In [ ]:
dset = read_mod.load_dataset(path_to_sol)

Clean data

> Extract post-shock states

In [ ]:
i_ps = dset["p"] > dset["p"][0,0]
for (k, v) in dset.items():
  if (k not in ("x", "t")):
    dset[k] = v[i_ps]

> Remove unphysical solutions

In [ ]:
dset["dT"] = dset["Th"] - dset["Te"]
i_dT = dset["dT"] > 0.0
for (k, v) in dset.items():
  if (k not in ("x", "t")):
    dset[k] = v[i_dT]

Compress mass fractions (i.e., distribution)
> Transform

In [ ]:
w = dset["W"]
wln = np.log(w)
wln_m = np.mean(wln, axis=0, keepdims=True)
wln_s = np.std(wln, axis=0, keepdims=True)
wln_n = (wln-wln_m)/wln_s

> Perform SVD

In [ ]:
yd = da.from_array(wln_n.T, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=rdim_max)
s = s.compute()
u = u.compute()

> Singular value decay

In [ ]:
# Normalized cumulative sum
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)
# Number of principal components
eps = 1e-5
rdim = np.where(cs > 1-eps)[0][0] + 1
print("Number of reduced dimensions:", rdim)

In [ ]:
plt_mod.cumenergy(
  cs[:rdim_max],
  figname=path_to_save + "/cum_en.pdf",
  save=True,
  show=False
)

> Check reconstruction error

In [ ]:
rom = rom_mod.ROM(
  phi=u[:,:rdim],
  xref=wln_m,
  xscale=wln_s
)

In [ ]:
wr = rom.autoencode(w)
err = 100*np.abs(wr-w)/(w+1e-15)
print("Max pointwise error: %.2f" % err.max(), "%")

> Save basis data

In [ ]:
rom.save(path_to_save)

Compute bounds

In [ ]:
bounds = {k: [dset[k].min(), dset[k].max()] for k in ("rho", "Th", "Te", "dT")}
bounds["Th"][0] = 5e3
bounds["Te"][0] = 3e3
bounds["dT"][0] = 0.0

In [ ]:
z = rom.encode(w)
for i in range(rdim):
  ki = f"z{i+1}"
  zi = z[:,i]
  dset[ki] = zi
  bounds[ki] = [zi.min(), zi.max()]

In [ ]:
with open(path_to_save+"/bounds.json", "w") as file:
  json.dump(bounds, file, indent=2)

Save dataset

In [ ]:
with open(path_to_save+"/dset.p", "wb") as file:
  pickle.dump(dset, file)